# 01 Video Data Exploration

Analyze the DFDC video dataset structure, dynamically discover available parts, and visualize class distribution.

### Key Goals:
- **Dynamic Discovery**: Scan for all `dfdc_train_part_*` folders.
- **Metadata Analysis**: Load `metadata.json` from each part to map labels.
- **Distribution**: Count REAL vs FAKE videos across detected parts.

In [1]:
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
import yaml
import json
import logging

# Load Config
with open("../../configs/video_config.yaml", "r") as f:
    config = yaml.safe_load(f)

DATA_PATH = Path(config["data"]["raw_path"])
print(f"Data Path: {DATA_PATH.resolve()}")

logging.basicConfig(level=config["logging"]["level"])
logger = logging.getLogger(__name__)

## 1. Dynamic Dataset Discovery

In [2]:
def discover_dataset_parts(base_path):
    """
    Dynamically discover all dfdc_train_part_* folders in the base_path.
    """
    base_path = Path(base_path)
    if not base_path.exists():
        print(f"Error: {base_path} does not exist.")
        return []
    
    parts = sorted([
        d for d in os.listdir(base_path)
        if d.startswith("dfdc_train_part_") and (base_path / d).is_dir()
    ], key=lambda x: int(x.split('_')[-1]))
    
    print(f"Found {len(parts)} dataset parts.")
    return [base_path / p for p in parts]

parts = discover_dataset_parts(DATA_PATH)
for p in parts:
    print(f" - {p.name}")

## 2. Collect Metadata & Labels

In [3]:
def collect_video_info(parts_paths):
    data = []
    for part_path in tqdm(parts_paths, desc="Scanning Parts"):
        metadata_path = part_path / "metadata.json"
        if not metadata_path.exists():
            print(f"Warning: No metadata.json in {part_path.name}")
            continue
            
        with open(metadata_path, "r") as f:
            metadata = json.load(f)
            
        for filename, info in metadata.items():
            video_path = part_path / filename
            if video_path.exists():
                data.append({
                    "path": str(video_path),
                    "part": part_path.name,
                    "label": info["label"],
                    "label_id": 1 if info["label"] == "FAKE" else 0
                })
    return pd.DataFrame(data)

df = collect_video_info(parts)
print(f"Total videos discovered: {len(df)}")
df.head()

## 3. Analyze Distribution

In [4]:
if not df.empty:
    dist = df["label"].value_counts()
    print("Class Distribution:")
    print(dist)
    
    dist.plot(kind="bar", color=['red', 'green'])
    plt.title("REAL vs FAKE Distribution")
    plt.ylabel("Count")
    plt.show()
    
    # Distribution per part
    part_dist = df.groupby(["part", "label"]).size().unstack(fill_value=0)
    part_dist.plot(kind="bar", stacked=True, figsize=(12, 6))
    plt.title("Distribution per Dataset Part")
    plt.show()
else:
    print("No data to analyze.")